# 01｜社區服務案件：三階提問實作

[![在 Colab 開啟](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/roberthsu2003/PythonForDataAnalysis/blob/master/%E7%9C%8B%E6%87%82%E8%B3%87%E6%96%99%E8%88%87%E9%81%B8%E5%B0%8D%E5%9C%96%E8%A1%A8/%E5%92%8CAI%E5%8D%94%E4%BD%9C%E7%9A%84%E4%B8%89%E9%9A%8E%E6%8F%90%E5%95%8F%E6%B3%95/01_%E7%A4%BE%E5%8D%80%E6%9C%8D%E5%8B%99%E6%A1%88%E4%BB%B6.ipynb)

> **比較 × 水平長條圖**｜課堂合成資料；一列代表 **一筆社區服務案件**。

| ① 看懂資料 | ② 一題一圖 | ③ 三輪追問 |
| :--- | :--- | :--- |
| 欄位與邊界 | 數字表與圖 | 核對數字 → 挑戰推論 → 修訂結論 |

**你負責提問與驗收，AI 負責寫程式。** 先執行第 0 格，依序將 prompt 貼給 Colab Gemini 或能讀取筆記本的 AI；參考程式供核對。第二階完成後，留在同一段對話，接著問第三階的三輪。

**核心問題：**2025 年已結案案件中，哪個服務類別的處理時數中位數最高？　｜　[下載資料集](./資料集/%E7%A4%BE%E5%8D%80%E6%9C%8D%E5%8B%99%E6%A1%88%E4%BB%B6.xlsx)

In [ ]:
# 0｜載入 xlsx，並沿用前面章節的繁體中文字型設定
from pathlib import Path
from urllib.request import urlretrieve
import glob
import os
import pandas as pd
import matplotlib.pyplot as plt

filename = '社區服務案件.xlsx'
file_path = next((p for p in [Path('資料集') / filename, Path(filename), Path('和AI協作的三階提問法/資料集') / filename] if p.exists()), None)
if file_path is None:
    file_path = Path(filename)
    urlretrieve('https://raw.githubusercontent.com/roberthsu2003/PythonForDataAnalysis/master/%E7%9C%8B%E6%87%82%E8%B3%87%E6%96%99%E8%88%87%E9%81%B8%E5%B0%8D%E5%9C%96%E8%A1%A8/%E5%92%8CAI%E5%8D%94%E4%BD%9C%E7%9A%84%E4%B8%89%E9%9A%8E%E6%8F%90%E5%95%8F%E6%B3%95/%E8%B3%87%E6%96%99%E9%9B%86/%E7%A4%BE%E5%8D%80%E6%9C%8D%E5%8B%99%E6%A1%88%E4%BB%B6.xlsx', file_path)
%pip install -q mplfonts
import mplfonts
from matplotlib import font_manager
from mplfonts import use_font
for font_file in glob.glob(os.path.join(os.path.dirname(mplfonts.__file__), 'fonts', '*.[ot]tf')):
    font_manager.fontManager.addfont(font_file)
use_font('Noto Sans CJK SC')
plt.rcParams['axes.unicode_minus'] = False
df = pd.read_excel(file_path, sheet_name='資料')
print('資料檔：', file_path, '；列數：', len(df))
display(df.head())

---

## ① 第一階｜看懂資料

**先自己判斷：**處理時數＝指標；受理日期、服務類別、案件狀態＝條件；案件編號＝標識。處理中的缺值不能當成 0 小時。

**複製給 AI**

```text
請讀取已載入的 df（社區服務案件.xlsx／工作表「資料」）。顯示前 5 列、總列數、欄位名稱、每欄缺值數；說明一列代表什麼，將欄位分成指標、條件、標識。檢查資料的時間範圍、重複或缺值，提出 2 個可以回答、1 個不能回答的問題並說明理由。先不要畫圖，也不要下結論。
```

**驗收：**筆數、缺值與欄位角色，都要和 xlsx 一致。

In [ ]:
print('資料列數：', len(df))
print('欄位：', list(df.columns))
display(df.head(5))
display(df.isna().sum().rename('缺值數').to_frame())

---

## ② 第二階｜一個問題、一張圖

**複製給同一段 AI 對話**

```text
延續第一階，回答「2025 年已結案案件中，哪個服務類別的處理時數中位數最高？」；這是比較問題。先篩選 2025 年、已結案且處理時數非缺值的案件。依服務類別計算案件數與處理時數中位數，製作由高到低的數字表。只畫一張水平長條圖（Horizontal Bar Chart）：Y 軸＝服務類別，X 軸＝處理時數中位數（小時），數值軸從 0 開始，長條標值。類別名稱長，所以選水平圖。標題、坐標軸用繁體中文，沿用前面章節的 mplfonts／Noto Sans CJK SC 字型設定。不要畫其他圖。
```

**驗收：**先核對數字表，再核對圖表類型、坐標軸、單位與中文字。

In [ ]:
closed = df.loc[(df['受理日期'].dt.year == 2025) & df['案件狀態'].eq('已結案') & df['處理時數'].notna()]
summary = closed.groupby('服務類別')['處理時數'].agg(案件數='size', 中位數_小時='median').sort_values('中位數_小時', ascending=False)
display(summary)
plot_data = summary.sort_values('中位數_小時')
fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(plot_data.index, plot_data['中位數_小時'], color=['#E07A5F' if x == summary.index[0] else '#8ABFCB' for x in plot_data.index])
ax.bar_label(bars, fmt='%.1f', padding=4)
ax.set(xlabel='處理時數中位數（小時）', ylabel='服務類別', title='2025 年已結案案件｜各類處理時數中位數')
ax.set_xlim(0, plot_data['中位數_小時'].max() * 1.18)
plt.tight_layout()
plt.show()

---

## ③ 第三階｜接著追問三輪

沿用第二階的數字表與圖，不要重開對話。

### 第 1 輪｜核對數字與排除資料

```text
逐列核對圖上中位數和數字表；列出原始案件數、已結案件數、排除的處理中案件數。確認缺值沒有被算成 0。若不一致，先修正圖和表，再回報修正。
```

In [ ]:
print('原始案件：', len(df), '；已結案且有時數：', len(closed), '；處理中：', df['案件狀態'].eq('處理中').sum())
assert summary['案件數'].sum() == len(closed)
assert df.loc[df['案件狀態'].eq('處理中'), '處理時數'].isna().all()

### 第 2 輪｜挑戰推論

```text
中位數較高能直接說該類服務效率最差嗎？指出至少兩項缺少的背景資料，並說明案件數與極端值如何影響解讀。不要重畫圖。
```

### 第 3 輪｜修訂三句結論

```text
用三句繁體中文寫結論：①中位數最高的類別與數字；②資料範圍、已結案件數和處理中案件的排除；③不能直接推論效率高低的理由。數字必須與剛才核對的表格一致。
```

> ✍️ **我的驗收紀錄：**哪個數字被核對？哪句推論被改寫？（雙擊此格記下來。）

---

[返回第 7 課](./README.md)　｜　[返回學習地圖](../README.md)